# Train Nôm glyph classifier (Kaggle GPU P100)

Train một **embedding-model phân biệt chữ Nôm** (ResNet-18 + ArcFace) để thay DINOv2 cho tín hiệu thị giác S3.
DINOv2 zero-shot đã chứng minh KHÔNG dùng được (cosine 0.91 giữa 2 chữ khác nhau, retrieval top-1 = 0%).

**Trước khi chạy:** đóng gói ở máy (`prepare_data.py` → `pack_for_kaggle.py`), upload `kaggle_pkg/` làm Kaggle Dataset,
rồi Add Input dataset đó vào notebook. Settings: **Accelerator = GPU P100**, **Internet = ON**.
Chi tiết: `KAGGLE.md`.

## 1. Kiểm GPU

In [ ]:
import torch, shutil, sys
assert shutil.which('nvidia-smi'), '🛑 Chưa bật GPU: Settings → Accelerator → GPU P100'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Trỏ ROOT tới dataset đã upload

Đổi `ROOT` cho khớp **slug** dataset của bạn (vd `nom-crops` → `/kaggle/input/nom-crops`).

In [ ]:
import os, sys, json, glob
# Tự dò dataset trong /kaggle/input có chứa index.csv (đỡ phải sửa tay)
cands = [os.path.dirname(p) for p in glob.glob('/kaggle/input/*/index.csv')]
ROOT = cands[0] if cands else '/kaggle/input/nom-crops'   # sửa nếu cần
assert os.path.exists(f'{ROOT}/index.csv'), f'Không thấy index.csv trong {ROOT} — kiểm slug/Add Input'
sys.path.insert(0, ROOT)
print('ROOT =', ROOT)
print(json.load(open(f'{ROOT}/stats.json')) if os.path.exists(f'{ROOT}/stats.json') else 'no stats.json')

## 3. Train (P100, ~1.5–3h / 35 epoch)

Checkpoint lưu ở `/kaggle/working/checkpoints/` (`best.pt`, `last.pt`). Giảm `--batch`/`--img` nếu OOM.
Nếu Internet OFF (không tải được weights ResNet-18): thêm `--no-pretrained`.

In [ ]:
!python {ROOT}/train.py \
    --root {ROOT} --index {ROOT}/index.csv --classes {ROOT}/classes.json \
    --out /kaggle/working/checkpoints \
    --epochs 35 --batch 256 --img 128 --workers 2

## 4. Nghiệm thu — so trực tiếp DINOv2

| Test | DINOv2 (hỏng) | Đạt khi |
|---|---|---|
| T2 separation (cùng−khác) | +0.012 | **≥ +0.20** |
| T3 retrieval top-1 | 0.0% | **≥ 80%** |

In [ ]:
!python {ROOT}/eval_discrim.py \
    --root {ROOT} --index {ROOT}/index.csv \
    --ckpt /kaggle/working/checkpoints/best.pt

## 5. Lưu & tải checkpoint

`best.pt` nằm trong **Output** (`/kaggle/working/checkpoints/`) → tải về, hoặc **Save Version** để giữ.

Bước tiếp ở máy: cắm `NomEncoder` (`infer.py`) vào `../visual_signal.py` thay DINOv2,
bật lại SILVER bằng `build_dataset.py --use-s3`.

In [ ]:
import os
ck = '/kaggle/working/checkpoints'
print('checkpoints:', os.listdir(ck) if os.path.exists(ck) else 'chưa có (chạy cell 3)')